# ULTRA Foundation Model: Coverage Blind Spot Test

**Hypothesis:** Even state-of-the-art foundation KG models (ULTRA) fail to detect novel relational contexts.

**Key Insight:** ULTRA uses entity embeddings derived from graph structure via NBFNet message passing. It does NOT track which (entity, relation) pairs were seen during training. Therefore:
- **Emerging entities** (low frequency): ULTRA might detect these (poor connectivity = less information)
- **Novel relational contexts** (entity seen, but not with this relation): ULTRA CANNOT detect these

**Expected Results:**
- Novel Context AUROC ~ 0.5 (random guessing)
- This confirms "the coverage blind spot is fundamental, not fixable by scale"

---

## 1. Setup: Install Dependencies

In [ ]:
# Install required packages
!pip install torch torch-geometric torch-scatter torch-sparse -q
!pip install easydict pyyaml tqdm pandas scikit-learn -q

# Clone ULTRA repository
!git clone https://github.com/DeepGraphLearning/ULTRA.git 2>/dev/null || echo "ULTRA already cloned"

# Download pretrained checkpoint
import os
os.makedirs('ULTRA/ckpts', exist_ok=True)
if not os.path.exists('ULTRA/ckpts/ultra_3g.pth'):
    !wget -q https://zenodo.org/record/8095626/files/ultra_3g.pth -O ULTRA/ckpts/ultra_3g.pth
    print("Downloaded ultra_3g.pth checkpoint")
else:
    print("Checkpoint already exists")

In [ ]:
# Add ULTRA to path and verify imports
import sys
sys.path.insert(0, 'ULTRA')

import torch
import numpy as np
from collections import defaultdict
from sklearn.metrics import roc_auc_score, average_precision_score
import time

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load ULTRA Model

In [ ]:
from ultra.models import Ultra
from ultra import datasets as ultra_datasets

def load_ultra_model(checkpoint_path, device):
    """Load pretrained ULTRA model with standard config."""
    model = Ultra(
        rel_model_cfg={
            'class': 'RelNBFNet',
            'input_dim': 64,
            'hidden_dims': [64, 64, 64, 64, 64, 64],
            'message_func': 'distmult',
            'aggregate_func': 'sum',
            'short_cut': True,
            'layer_norm': True
        },
        entity_model_cfg={
            'class': 'EntityNBFNet',
            'input_dim': 64,
            'hidden_dims': [64, 64, 64, 64, 64, 64],
            'message_func': 'distmult',
            'aggregate_func': 'sum',
            'short_cut': True,
            'layer_norm': True
        }
    )
    
    state = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(state['model'])
    model = model.to(device)
    model.eval()
    return model

# Load model
print("Loading ULTRA model...")
model = load_ultra_model('ULTRA/ckpts/ultra_3g.pth', device)
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Load FB15k-237 Dataset

In [ ]:
# Load FB15k-237 in ULTRA format
print("Loading FB15k-237 dataset...")
os.makedirs('kg-datasets', exist_ok=True)
dataset = ultra_datasets.FB15k237('kg-datasets')

train_data = dataset[0]
valid_data = dataset[1]
test_data = dataset[2]

print(f"\nDataset statistics:")
print(f"  Entities: {train_data.num_nodes:,}")
print(f"  Relations: {train_data.num_relations}")
print(f"  Train triples: {train_data.edge_index.size(1):,}")
print(f"  Test triples: {test_data.target_edge_index.size(1):,}")

## 4. Build Coverage Matrix and Split Test Triples

We categorize test triples into:
1. **Emerging**: At least one entity has low frequency in training (bottom 25th percentile)
2. **Novel Context**: Both entities are well-seen, but at least one was never seen with this relation
3. **In-Distribution (ID)**: Both entities have been seen with this relation before

In [ ]:
def build_coverage_matrix(edge_index, edge_type, num_entities, num_relations):
    """
    Build coverage matrix: coverage[e, r] = 1 if entity e was seen with relation r.
    This is the key structural signal that ULTRA cannot track.
    """
    coverage = np.zeros((num_entities, num_relations))
    heads = edge_index[0].cpu().numpy()
    tails = edge_index[1].cpu().numpy()
    rels = edge_type.cpu().numpy()
    
    for h, t, r in zip(heads, tails, rels):
        coverage[h, r] = 1.0
        coverage[t, r] = 1.0
    
    return coverage

def compute_entity_frequency(edge_index):
    """Compute how often each entity appears in training."""
    freq = defaultdict(int)
    heads = edge_index[0].cpu().numpy()
    tails = edge_index[1].cpu().numpy()
    for h, t in zip(heads, tails):
        freq[h] += 1
        freq[t] += 1
    return freq

# Build coverage matrix from training data
print("Building coverage matrix...")
coverage = build_coverage_matrix(
    train_data.edge_index, 
    train_data.edge_type,
    train_data.num_nodes,
    train_data.num_relations
)

# Compute entity frequencies
freq = compute_entity_frequency(train_data.edge_index)
freq_values = list(freq.values())
freq_threshold = np.percentile(freq_values, 25)

print(f"\nCoverage matrix shape: {coverage.shape}")
print(f"Entity frequency threshold (25th percentile): {freq_threshold}")
print(f"Coverage density: {coverage.sum() / coverage.size:.4f}")

In [ ]:
def categorize_test_triples(test_data, freq, coverage, threshold):
    """
    Categorize test triples into emerging, novel_context, and ID.
    
    Returns lists of indices for each category.
    """
    heads = test_data.target_edge_index[0].cpu().numpy()
    tails = test_data.target_edge_index[1].cpu().numpy()
    rels = test_data.target_edge_type.cpu().numpy()
    
    emerging_idx = []
    novel_ctx_idx = []
    id_idx = []
    
    for i, (h, t, r) in enumerate(zip(heads, tails, rels)):
        h_freq = freq.get(h, 0)
        t_freq = freq.get(t, 0)
        
        # Emerging: at least one entity has low frequency
        if h_freq <= threshold or t_freq <= threshold:
            emerging_idx.append(i)
        # Novel context: both entities well-seen, but missing (entity, relation) coverage
        elif coverage[h, r] == 0 or coverage[t, r] == 0:
            novel_ctx_idx.append(i)
        # ID: both entities seen with this relation
        else:
            id_idx.append(i)
    
    return emerging_idx, novel_ctx_idx, id_idx

# Categorize test triples
print("Categorizing test triples...")
emerging_idx, novel_ctx_idx, id_idx = categorize_test_triples(
    test_data, freq, coverage, freq_threshold
)

print(f"\nTest triple split:")
print(f"  Emerging:      {len(emerging_idx):,} ({100*len(emerging_idx)/len(emerging_idx+novel_ctx_idx+id_idx):.1f}%)")
print(f"  Novel Context: {len(novel_ctx_idx):,} ({100*len(novel_ctx_idx)/len(emerging_idx+novel_ctx_idx+id_idx):.1f}%)")
print(f"  In-Distribution: {len(id_idx):,} ({100*len(id_idx)/len(emerging_idx+novel_ctx_idx+id_idx):.1f}%)")

## 5. Score Test Triples with ULTRA

ULTRA outputs a score for each triple. Higher score = more likely to be true.
We use **negative score as uncertainty** (like energy-based methods).

In [ ]:
@torch.no_grad()
def score_triples_ultra(model, data, batch_size=64):
    """
    Score all test triples using ULTRA.
    Returns array of scores (higher = more confident).
    """
    model.eval()
    data = data.to(device)
    
    heads = data.target_edge_index[0]
    tails = data.target_edge_index[1]
    rels = data.target_edge_type
    
    n_triples = len(heads)
    scores = []
    
    print(f"Scoring {n_triples:,} test triples...")
    t0 = time.time()
    
    for i in range(0, n_triples, batch_size):
        h = heads[i:i+batch_size]
        t = tails[i:i+batch_size]
        r = rels[i:i+batch_size]
        
        # ULTRA expects batch of shape (bs, 1+num_negs, 3)
        # Format: [head, tail, relation]
        batch = torch.stack([h, t, r], dim=-1).unsqueeze(1)  # (bs, 1, 3)
        
        try:
            score = model(data, batch)  # (bs, 1)
            scores.append(score.squeeze(-1).cpu())
        except Exception as e:
            print(f"Error at batch {i}: {e}")
            scores.append(torch.zeros(len(h)))
        
        if (i + batch_size) % 5000 < batch_size:
            elapsed = time.time() - t0
            progress = (i + batch_size) / n_triples
            eta = elapsed / progress - elapsed if progress > 0 else 0
            print(f"  Progress: {i+batch_size:,}/{n_triples:,} ({100*progress:.1f}%) - ETA: {eta:.0f}s")
    
    elapsed = time.time() - t0
    print(f"Scoring completed in {elapsed:.1f}s")
    
    return torch.cat(scores).numpy()

# Score all test triples
all_scores = score_triples_ultra(model, test_data, batch_size=128)
print(f"\nScore statistics:")
print(f"  Min: {all_scores.min():.4f}")
print(f"  Max: {all_scores.max():.4f}")
print(f"  Mean: {all_scores.mean():.4f}")
print(f"  Std: {all_scores.std():.4f}")

## 6. Compute OOD Detection AUROC

**Key Question:** Can ULTRA's confidence distinguish OOD triples from ID triples?

We use **negative score as uncertainty** (higher uncertainty = more likely OOD).

In [ ]:
def compute_ood_auroc(scores, ood_idx, id_idx, label='OOD'):
    """
    Compute AUROC for distinguishing OOD from ID using negative scores as uncertainty.
    
    Higher uncertainty (lower score) should indicate OOD.
    AUROC > 0.5 means the method can detect OOD.
    AUROC ~ 0.5 means random guessing (blind spot).
    """
    if len(ood_idx) < 50 or len(id_idx) < 50:
        print(f"  {label}: Insufficient samples (OOD={len(ood_idx)}, ID={len(id_idx)})")
        return None
    
    # Uncertainty = negative score (lower score = higher uncertainty)
    ood_uncertainty = -scores[ood_idx]
    id_uncertainty = -scores[id_idx]
    
    # Labels: 1 for OOD, 0 for ID
    labels = np.concatenate([np.zeros(len(id_uncertainty)), np.ones(len(ood_uncertainty))])
    uncertainties = np.concatenate([id_uncertainty, ood_uncertainty])
    
    auroc = roc_auc_score(labels, uncertainties)
    aupr = average_precision_score(labels, uncertainties)
    
    # Also compute score statistics for interpretation
    ood_mean = scores[ood_idx].mean()
    id_mean = scores[id_idx].mean()
    
    print(f"  {label}:")
    print(f"    AUROC: {auroc:.4f}")
    print(f"    AUPR: {aupr:.4f}")
    print(f"    Mean score (OOD): {ood_mean:.4f}")
    print(f"    Mean score (ID): {id_mean:.4f}")
    print(f"    Score gap: {id_mean - ood_mean:.4f}")
    
    return auroc, aupr

print("="*60)
print("ULTRA OOD DETECTION RESULTS")
print("="*60)
print()

# 1. Overall: (Emerging + Novel Context) vs ID
print("1. Overall OOD Detection (Emerging + Novel Context vs ID):")
ood_idx = emerging_idx + novel_ctx_idx
overall_result = compute_ood_auroc(all_scores, ood_idx, id_idx, 'Overall')
print()

# 2. Emerging vs ID
print("2. Emerging Entity Detection (Emerging vs ID):")
emerging_result = compute_ood_auroc(all_scores, emerging_idx, id_idx, 'Emerging')
print()

# 3. Novel Context vs ID (THIS IS THE KEY TEST)
print("3. Novel Relational Context Detection (Novel Context vs ID):")
print("   >>> This is the COVERAGE BLIND SPOT test <<<")
novel_ctx_result = compute_ood_auroc(all_scores, novel_ctx_idx, id_idx, 'Novel Context')

## 7. Summary and Interpretation

In [ ]:
print("="*70)
print("SUMMARY: ULTRA Foundation Model Coverage Blind Spot Test")
print("="*70)
print()
print(f"Dataset: FB15k-237")
print(f"Model: ULTRA (pretrained on 3 graphs)")
print()
print(f"Test split sizes:")
print(f"  Emerging:      {len(emerging_idx):,}")
print(f"  Novel Context: {len(novel_ctx_idx):,}")
print(f"  ID:            {len(id_idx):,}")
print()
print("Results (AUROC - higher is better, 0.5 is random):")
print("-" * 50)

if overall_result:
    print(f"  Overall OOD (Em+NC vs ID):    {overall_result[0]:.4f}")
if emerging_result:
    print(f"  Emerging vs ID:               {emerging_result[0]:.4f}")
if novel_ctx_result:
    print(f"  Novel Context vs ID:          {novel_ctx_result[0]:.4f}  << KEY RESULT")

print()
print("Interpretation:")
print("-" * 50)

if novel_ctx_result:
    nc_auroc = novel_ctx_result[0]
    if nc_auroc < 0.55:
        print("CONFIRMED: ULTRA has the coverage blind spot!")
        print(f"Novel context AUROC = {nc_auroc:.4f} (near random)")
        print("")
        print("ULTRA cannot distinguish between:")
        print("  - ID: entity seen with this relation")
        print("  - OOD: entity seen but NEVER with this relation")
        print("")
        print("This confirms: the blind spot is FUNDAMENTAL,")
        print("not fixable by scale or foundation models.")
    elif nc_auroc < 0.65:
        print(f"PARTIAL: ULTRA shows weak novel context detection ({nc_auroc:.4f})")
        print("Some signal exists but far from reliable.")
    else:
        print(f"UNEXPECTED: ULTRA shows meaningful novel context detection ({nc_auroc:.4f})")
        print("This contradicts our hypothesis - needs investigation.")

## 8. Comparison with CAGP (for Paper)

The results above should be compared with CAGP from our main experiments:

| Method | Emerging AUROC | Novel Ctx AUROC | Overall AUROC |
|--------|---------------|-----------------|---------------|
| ULTRA  | (from above)  | (from above)    | (from above)  |
| U_sem (GP-KGE) | 0.81 | ~0.5 | 0.59 |
| U_str (Coverage) | 0.78 | 0.94 | 0.94 |
| CAGP (Ours) | 0.89 | 0.97 | 0.97 |

**Key insight:** ULTRA, despite being a foundation model trained on multiple graphs,
should show similar limitations to U_sem on novel relational contexts.

In [ ]:
# Save results for paper
import json

results = {
    'model': 'ULTRA (ultra_3g)',
    'dataset': 'FB15k-237',
    'split_sizes': {
        'emerging': len(emerging_idx),
        'novel_context': len(novel_ctx_idx),
        'id': len(id_idx)
    },
    'auroc': {
        'overall': overall_result[0] if overall_result else None,
        'emerging': emerging_result[0] if emerging_result else None,
        'novel_context': novel_ctx_result[0] if novel_ctx_result else None
    },
    'aupr': {
        'overall': overall_result[1] if overall_result else None,
        'emerging': emerging_result[1] if emerging_result else None,
        'novel_context': novel_ctx_result[1] if novel_ctx_result else None
    },
    'frequency_threshold': float(freq_threshold)
}

with open('ultra_ood_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to ultra_ood_results.json")
print()
print(json.dumps(results, indent=2))

## 9. Additional Analysis: Score Distributions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot score distributions
bins = 50

# 1. Emerging vs ID
ax = axes[0]
ax.hist(all_scores[emerging_idx], bins=bins, alpha=0.7, label='Emerging', density=True)
ax.hist(all_scores[id_idx], bins=bins, alpha=0.7, label='ID', density=True)
ax.set_xlabel('ULTRA Score')
ax.set_ylabel('Density')
ax.set_title(f'Emerging vs ID\nAUROC={emerging_result[0]:.3f}' if emerging_result else 'Emerging vs ID')
ax.legend()

# 2. Novel Context vs ID (KEY)
ax = axes[1]
ax.hist(all_scores[novel_ctx_idx], bins=bins, alpha=0.7, label='Novel Context', density=True, color='orange')
ax.hist(all_scores[id_idx], bins=bins, alpha=0.7, label='ID', density=True, color='green')
ax.set_xlabel('ULTRA Score')
ax.set_ylabel('Density')
ax.set_title(f'Novel Context vs ID (BLIND SPOT TEST)\nAUROC={novel_ctx_result[0]:.3f}' if novel_ctx_result else 'Novel Context vs ID')
ax.legend()

# 3. Overall
ax = axes[2]
ood_scores = all_scores[emerging_idx + novel_ctx_idx]
ax.hist(ood_scores, bins=bins, alpha=0.7, label='OOD (Em+NC)', density=True, color='red')
ax.hist(all_scores[id_idx], bins=bins, alpha=0.7, label='ID', density=True, color='green')
ax.set_xlabel('ULTRA Score')
ax.set_ylabel('Density')
ax.set_title(f'Overall OOD vs ID\nAUROC={overall_result[0]:.3f}' if overall_result else 'Overall OOD vs ID')
ax.legend()

plt.tight_layout()
plt.savefig('ultra_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved figure to ultra_score_distributions.png")

---

## Conclusion

This notebook tests whether ULTRA, a state-of-the-art foundation model for knowledge graphs, suffers from the same **coverage blind spot** that affects traditional KGE methods.

**Expected finding:** Novel Context AUROC ~ 0.5, confirming that ULTRA cannot detect when an entity appears in a relational context it was never seen in during training.

**Implication for the paper:** This supports our claim that the coverage blind spot is **fundamental to embedding-based methods** and cannot be fixed by:
- More parameters
- Foundation model pretraining
- Graph neural network architectures

The solution requires **explicit coverage tracking** as in our CAGP method.